<a href="https://colab.research.google.com/github/stephenchuong/data-cleaning-project-with-online-sales-dataset/blob/main/retail_cleaned_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**load the dataset**




In [485]:
#import functions
import pandas as pd
import numpy as np
data = pd.read_csv ('/content/Online_Retail_Dataset - Sheet1.csv')
print("data overview:\n",data.head())
print("data description:\n",data.describe())
#This gives us an idea of what to clean, for example "OrderDate", the dates,...

data overview:
   InvoiceNo CustomerName ProductCategory OrderDate ShippingDate   Price  \
0  INV-2427        Frank     Electronics  ########     7/2/2022  340.02   
1  INV-5268          Eve           Books  ########     ########  382.57   
2  INV-7221      Charlie            Toys  ########     ########   36.20   
3  INV-9730        Diana        Clothing  ########     ########  402.89   
4  INV-2330          Bob            Toys  ########     1/9/2022  234.17   

   Quantity Country  TotalAmount  
0       5.0     USA      1700.10  
1       2.0  France       765.14  
2      10.0      UK       362.00  
3       9.0     USA      3626.01  
4       2.0  Canada       468.34  
data description:
              Price     Quantity  TotalAmount
count  1515.000000  1513.000000  1511.000000
mean    247.748805     5.432915  1354.293402
std     141.643167     2.910645  1130.540830
min       5.020000     1.000000     7.000000
25%     125.485000     3.000000   415.065000
50%     244.240000     5.000000  1

**Check for null values**

In [486]:
#check for null values
missing_data = data.isnull().sum()
print(missing_data)
# There are 6 null values in InvoiceNO, 3 in CustomerName,...

InvoiceNo          6
CustomerName       3
ProductCategory    8
OrderDate          3
ShippingDate       5
Price              5
Quantity           7
Country            4
TotalAmount        9
dtype: int64


**Handle missing data**

There are a numbr of ways to handle missing data : imputation, dropping null values, ...
In this case, we will drop null values in the collumns that are not critical to the analysis, these are: "CustomerName", "ProduceCategory" and "Price".
The values to be dropped depend on the task, for this given task these values are not necessary because the final goal is to simply plot the data

In [487]:
data.columns
#Check for the spelling of the columns

Index(['InvoiceNo', 'CustomerName', 'ProductCategory', 'OrderDate',
       'ShippingDate', 'Price', 'Quantity', 'Country', 'TotalAmount'],
      dtype='object')

In [488]:
data = data.dropna(subset=['CustomerName','ProductCategory','Price'])
print(data.isnull().sum())
#The null values have been droppped successfully

print("\nnull values have been dropped!!")

InvoiceNo          6
CustomerName       0
ProductCategory    0
OrderDate          3
ShippingDate       5
Price              0
Quantity           7
Country            4
TotalAmount        9
dtype: int64

null values have been dropped!!


In [489]:
#Missing values in the "Country" collumn will be filled by "unknown"
data['Country'] = data['Country'].fillna('Unknown')

**Remove duplicates**

In [490]:
duplicates = data.duplicated()
print("dupes total:", duplicates.sum())
#This means that there are a total of 19 duplicates in our dataset

dupes total: 19


In [491]:
#We can also check for duplicates in specific columns
duplicates_InvoiceNo = data.duplicated(subset='InvoiceNo')
print("\ndupes in invoiceno:", duplicates_InvoiceNo.sum())
#This means that there are a total of 144 duplicates in the InvoiceNo column, however, this is not a problem


dupes in invoiceno: 144


In [492]:
data = data.drop_duplicates()
print(f"number of dupes remaining: {data.duplicated().sum()}")

number of dupes remaining: 0


**Fix data types**

In this step, we primarily want to ensure that the date colmns are formated correctly, 03/05/2022 for example could either mean third of May or Fifth of March without forcing the column to follow a certain date format. The date format for this dataset will be m-d-Y

If a collumn has whole numbers( "Quantity" or "InvoiceNo") then it would be best to format them as intergers.

In [493]:
#Firstly let's check for the data type of the columns
print(data.dtypes)

InvoiceNo           object
CustomerName        object
ProductCategory     object
OrderDate           object
ShippingDate        object
Price              float64
Quantity           float64
Country             object
TotalAmount        float64
dtype: object


In [494]:
from re import error
#Convert the 'date' columns
data['OrderDate'] = pd.to_datetime(data['OrderDate'],errors='coerce')
data['ShippingDate'] = pd.to_datetime(data['ShippingDate'], errors= 'coerce')
#Convert the 'numerical' columns
data['Quantity'] = pd.to_numeric(data['Quantity'], errors = 'coerce')

print(data.dtypes)



InvoiceNo                  object
CustomerName               object
ProductCategory            object
OrderDate          datetime64[ns]
ShippingDate       datetime64[ns]
Price                     float64
Quantity                  float64
Country                    object
TotalAmount               float64
dtype: object


/tmp/ipykernel_2913/3593411327.py:3: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



In [495]:
# I think "Quantity" cannot be a float type so i will also convert it into integers, it makes more sense to me
data['Quantity'] = data['Quantity'].astype('Int64')
print(data.dtypes)

InvoiceNo                  object
CustomerName               object
ProductCategory            object
OrderDate          datetime64[ns]
ShippingDate       datetime64[ns]
Price                     float64
Quantity                    Int64
Country                    object
TotalAmount               float64
dtype: object


**Detect outliers**

In this step we want to remove any values that are above or below 75% or 25%, respectively of the middle 50% average of our prices.

We will use the IRL technique to remove ouliers
IQR is short for interquartile range and it measures the spread of the middle 50% of the prices.

Q1 means quarter 1 and this is the 25th percentile ( the lower quarter of my prices) and the rest 75% are higher than this number.

Q3 means quarter 3 and this is the 75th percentile of my prices. The rest 25% are higher than this nuber.

Lower and upper bounds are just the range where any prices that fall outside the middle 50% of my prices would be considered as an outlier. The standard rule is to multiply the IQR by 1.5.

Any price lower than the lower bound ( Q1 – IQR*1.5) gets cut
Any price higher than the upper bound (Q3 + IQR*1.5) gets cut.


In [496]:
Q1 = data['Price'].quantile(0.25)
Q3 = data['Price'].quantile(0.75)
IQR = Q3-Q1
Lower_bound = Q1 - 1.5*IQR
Upper_bound = Q3 + 1.5*IQR


In [497]:
#After setting the parameters, now we will drop any values that fall outside this parameters
data = data[(data['Price'] >= Lower_bound) & (data['Price'] <= Upper_bound)]
print(data)
#verify that the outliders have been succesfully removed
print("\n**Outliers removed!**")

     InvoiceNo CustomerName ProductCategory  OrderDate ShippingDate   Price  \
0     INV-2427        Frank     Electronics        NaT   2022-07-02  340.02   
1     INV-5268          Eve           Books        NaT          NaT  382.57   
2     INV-7221      Charlie            Toys        NaT          NaT   36.20   
3     INV-9730        Diana        Clothing        NaT          NaT  402.89   
4     INV-2330          Bob            Toys        NaT   2022-01-09  234.17   
...        ...          ...             ...        ...          ...     ...   
1495  INV-4646        Alice            Toys 2021-07-09          NaT  105.44   
1496  INV-2897          Bob       Groceries        NaT   2022-08-02  273.79   
1497  INV-3602        Alice        Clothing 2021-03-07          NaT  365.22   
1498  INV-6712        Alice           Books        NaT          NaT   31.68   
1499  INV-9467        Frank           Books        NaT   2022-09-06  447.57   

      Quantity    Country  TotalAmount  
0         

**Standardise column names**

Standardising column names help our analysis by making data easier to work with in ways suchh as improving consitency, for example making all collumn names follow the same format and avoiding mistakes like typos, or mismatch name in code. Also simplify codes.

In [498]:
data.columns = data.columns.str.strip().str.lower().str.replace(' ', '_')
print(data.columns)

Index(['invoiceno', 'customername', 'productcategory', 'orderdate',
       'shippingdate', 'price', 'quantity', 'country', 'totalamount'],
      dtype='object')


**Save the cleaned dataset**

In [499]:
cleaned_data = data
cleaned_data.to_csv('cleaned_data.csv', index=False)

**PLot the dataset**

We are going to do 3 visualizations: a bar graph, a histogram and a line graph

In [500]:
import plotly.express as px

# PLot a hitogram of the "price" column
fig = px.histogram(cleaned_data, x='price', nbins= 30, color_discrete_sequence=  ['grey'] , title='Distribution of Prices')
fig.show()

#Update the plot for a vibrant layout
fig.update_layout(
    xaxis_title='Price',
    yaxis_title='Frenquency',
    title_font = dict(size = 20 , color= 'red')
)
fig.show()


In [501]:
#Plot a bar chart
country_sales = data["country"].value_counts().reset_index()
country_sales.columns = ["country","count of sales"]

fig = px.bar(
    x = "country",
    y = "count of sales",
    data_frame = country_sales,
    title = "Sales per country",
    color = "country"
)
fig.show()

#Update the plot for a vibrant layout
fig.update_layout(
    xaxis_title='Country',
    yaxis_title='Sales',
    title_font = dict(size = 20 , color= 'pink')
)
fig.show()

In [504]:
#Plot a line graph

#plot data by "orderdate" and "totalamount"

sales_trend = data.groupby("orderdate")["totalamount"].sum().reset_index()

fig = px.line(
    sales_trend,
    x = "orderdate",
    y = "totalamount",
    title = "total sales over time",
    labels = {"orderdate": "date " , "totalamount": "total sales"},
    line_shape = "spline",
    color_discrete_sequence=  ['green']
)
fig.show()

#update for a vibrant layout
fig.update_layout(
    xaxis_title = "date",
    yaxis_title = "Total Sales",
    title_font = dict(size = 20 ,color= 'purple'),
    template = "plotly_white"
)
fig.show()

In [506]:
final_cleaned_data = data
final_cleaned_data.to_csv('final_cleaned_data.csv', index=False)